In [1]:
!wget 'https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw3/id_lr_3/1/1.csv'

--2025-05-13 14:22:14--  https://raw.githubusercontent.com/modernpacifist/engineering-of-machine-learning-urfu/refs/heads/master/data-storing-and-management/hw3/id_lr_3/1/1.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 230896 (225K) [text/plain]
Saving to: ‘1.csv’

1.csv               100%[===================>] 225.48K  --.-KB/s    in 0.03s   

2025-05-13 14:22:15 (8.10 MB/s) - ‘1.csv’ saved [230896/230896]



In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, skewness, kurtosis
from pyspark.sql.types import DoubleType
import pandas as pd
import json

In [3]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("HW3_DSAM") \
    .getOrCreate()

In [4]:
# 1. Read the CSV file into a Spark DataFrame
df = spark.read.csv("1.csv", header=True, inferSchema=True)
print("Original DataFrame schema:")
df.printSchema()

Original DataFrame schema:
root
 |-- feature_1: double (nullable = true)
 |-- feature_2: double (nullable = true)
 |-- feature_3: double (nullable = true)
 |-- feature_4: double (nullable = true)
 |-- feature_5: double (nullable = true)
 |-- feature_6: double (nullable = true)
 |-- feature_7: double (nullable = true)
 |-- feature_8: double (nullable = true)
 |-- feature_9: double (nullable = true)
 |-- id: integer (nullable = true)
 |-- target: integer (nullable = true)



In [5]:
# 2. Convert feature_3 column to numeric type
df = df.withColumn("feature_3", col("feature_3").cast(DoubleType()))
print("\nDataFrame schema after type conversion:")
df.printSchema()


DataFrame schema after type conversion:
root
 |-- feature_1: double (nullable = true)
 |-- feature_2: double (nullable = true)
 |-- feature_3: double (nullable = true)
 |-- feature_4: double (nullable = true)
 |-- feature_5: double (nullable = true)
 |-- feature_6: double (nullable = true)
 |-- feature_7: double (nullable = true)
 |-- feature_8: double (nullable = true)
 |-- feature_9: double (nullable = true)
 |-- id: integer (nullable = true)
 |-- target: integer (nullable = true)



In [6]:
# 3. Display statistical characteristics of feature_3 column
print("\nStatistical characteristics of feature_3:")
stats = df.select("feature_3").summary().toPandas()
print(stats)


Statistical characteristics of feature_3:
  summary          feature_3
0   count               1209
1    mean  64.96905615277518
2  stddev  14.23633075684728
3     min  41.23606562713802
4     25%  52.09442943540296
5     50%  65.35780386237494
6     75%  77.60114416666455
7     max  89.19369362450949


In [7]:
# 4. Calculate skewness and kurtosis coefficients
skew = df.select(skewness("feature_3")).collect()[0][0]
kurt = df.select(kurtosis("feature_3")).collect()[0][0]
print(f"\nSkewness of feature_3: {skew}")
print(f"Kurtosis of feature_3: {kurt}")


Skewness of feature_3: -0.0039003026866904146
Kurtosis of feature_3: -1.2289392104241366


In [8]:
# 5. Write statistical characteristics and additional metrics to JSON file
# Create a dictionary to store the results
stats_dict = {}

# Add basic statistics from summary
for row in stats.itertuples():
    if row.summary in ['count', 'mean', 'stddev', 'min', 'max']:
        stats_dict[row.summary] = float(row.feature_3) if row.summary != 'count' else int(float(row.feature_3))

# Add skewness and kurtosis
stats_dict['skewness'] = float(skew)
stats_dict['kurtosis'] = float(kurt)

# Save to JSON file
with open("feature_3_statistics.json", "w") as f:
    json.dump(stats_dict, f, indent=4)

print("\nStatistics saved to feature_3_statistics.json")


Statistics saved to feature_3_statistics.json
